# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [16]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [11]:
# atom() is provided in sudoku_solver.py and imported above.

## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [12]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.

### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [18]:
# Run each experiment in a separate process and stop it after 10 seconds to avoid blocking the notebook.
import subprocess
import sys
from pathlib import Path

experiment_code = r"""
import json, sys, time
from sudoku_solver import atom, build_general_kb
from logic_ import pl_resolution, tt_entails, associate

with open('puzzles.json') as f:
    raw = json.load(f)
p = raw['puzzles'][int(sys.argv[2])]
givens = {tuple(map(int, k.split('_'))): v for k, v in p['givens'].items()}
kb = build_general_kb(raw['n'], raw['box_h'], raw['box_w'], givens)
query = atom('Is', 1, 1, 1)
start = time.perf_counter()
if sys.argv[1] == 'resolution':
    result = pl_resolution(kb, query)
else:
    result = tt_entails(associate('&', kb.clauses), query)
print(f'Result: {result}; inference time: {time.perf_counter() - start:.3f}s')
"""

experiment_observations = []
puzzle_index = next(i for i, p in enumerate(puzzle_pool) if p is puzzle)
for method in ('resolution', 'truth_table'):
    start = time.perf_counter()
    try:
        process = subprocess.run(
            [sys.executable, '-B', '-c', experiment_code, method, str(puzzle_index)],
            cwd=Path(sudoku_solver.__file__).parent,
            capture_output=True, text=True, timeout=10,
        )
        result = process.stdout.strip() if process.returncode == 0 else process.stderr.strip()
        observation = f"{method}: {result}"
    except subprocess.TimeoutExpired:
        elapsed = time.perf_counter() - start
        observation = (f"{method}: no answer before the 10-second process limit; "
                       f"process stopped (total elapsed {elapsed:.2f}s).")
    experiment_observations.append(observation)
    print(observation)

resolution: no answer before the 10-second process limit; process stopped (total elapsed 10.12s).
truth_table: no answer before the 10-second process limit; process stopped (total elapsed 10.01s).


### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


The experiment above asks whether cell (1, 1) has value 1 and limits each separate process to 10 seconds, including setup. See the actual output above for the observed result. A timeout means no answer was obtained within this budget; it does **not** mean the query is false.

For this 9 × 9 general KB there are 729 propositional atoms. Truth-table enumeration has a worst-case search space of $2^{729}$ assignments. The supplied resolution implementation initially enumerates all pairs of clauses and may generate many additional resolvents. The first puzzle has 11,775 KB clauses. Resolution adds the negated atomic query before enumerating pairs, so its initial list has 11,776 entries and 69,331,200 pairs. These are different sources of computational cost.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [17]:
# Time both full-grid solvers on the same puzzle, each using a fresh knowledge base.
start = time.perf_counter()
solved_fc = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.perf_counter() - start

start = time.perf_counter()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.perf_counter() - start

assert solved_fc == puzzle['solution']
assert solved_bc == puzzle['solution']
print(f"Forward chaining:  {fc_time:.4f}s")
print(f"Backward chaining: {bc_time:.4f}s")
print("Both solutions match the reference. Solved grid:")
for r in range(1, n + 1):
    print(' '.join(str(solved_fc[(r, c)]) for c in range(1, n + 1)))

print("\nTiming includes KB construction. FC uses the provided pl_fc_entails "
      "with indexed rule lookup; BC caches proved goals within each KB.")

Forward chaining:  5.9964s
Backward chaining: 0.2081s
Both solutions match the reference. Solved grid:
1 3 6 9 8 5 4 7 2
2 9 4 3 1 7 8 6 5
5 8 7 2 6 4 1 3 9
7 5 1 4 9 6 2 8 3
3 6 9 8 7 2 5 1 4
4 2 8 5 3 1 7 9 6
8 1 2 6 4 9 3 5 7
6 4 3 7 5 8 9 2 1
9 7 5 1 2 3 6 4 8

Timing includes KB construction. FC uses the provided pl_fc_entails with indexed rule lookup; BC caches proved goals within each KB.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# solution only used for check；The solver and inference functions only read givens.
print("givens | FC seconds | BC seconds | BC query checks")
verification_results = []
for item in puzzle_pool:
    start = time.perf_counter()
    answer_fc = solve_full_grid_fc(n, box_h, box_w, item['givens'])
    elapsed_fc = time.perf_counter() - start
    start = time.perf_counter()
    answer_bc = solve_full_grid_bc(n, box_h, box_w, item['givens'])
    elapsed_bc = time.perf_counter() - start
    assert answer_fc == item['solution']
    assert answer_bc == item['solution']

    check_kb = build_definite_kb(n, box_h, box_w, item['givens'])
    checked = 0
    for (r, c), correct_value in item['solution'].items():
        for v in range(1, n + 1):
            actual = pl_bc_entails(check_kb, atom('Is', r, c, v))
            assert actual == (v == correct_value), (item['given_count'], r, c, v)
            checked += 1

    verification_results.append((item['given_count'], elapsed_fc, elapsed_bc, checked))
    print(f"{item['given_count']:6d} | {elapsed_fc:10.4f} | {elapsed_bc:10.4f} | {checked} passed")

print("All five puzzles solved by both methods; all 3,645 BC queries passed.")

givens | FC seconds | BC seconds | BC query checks


    30 |     5.8093 |     0.1845 | 729 passed


    33 |     5.6295 |     0.1732 | 729 passed


    36 |     5.3074 |     0.1630 | 729 passed


    39 |     4.7268 |     0.1787 | 729 passed


    42 |     4.3822 |     0.1602 | 729 passed
All five puzzles solved by both methods; all 3,645 BC queries passed.


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Answer:**

Let $I_{rcv}$ denote the proposition that cell $(r,c)$ contains value $v$. In the general representation, I use only the `Is` symbols. Logical negation is written as `~atom('Is', r, c, v)`, so a separate `Not` symbol is unnecessary. The knowledge base is a conjunction of clauses stored in a `PropKB`.

For every cell, the clause $I_{rc1} \lor \cdots \lor I_{rcn}$ requires at least one value. For each pair of different values $a<b$, I add $\neg I_{rca} \lor \neg I_{rcb}$, which prevents that cell from taking both values. Together these constraints require exactly one value per cell. For each value $v$, I also add $\neg I_{rc_1v} \lor \neg I_{rc_2v}$ for every pair of different columns in a row, and $\neg I_{r_1cv} \lor \neg I_{r_2cv}$ for every pair of different rows in a column. Within each box, every pair of distinct cells receives the same kind of exclusion clause. These clauses prohibit repeated values in a row, column or box. Each given $(r,c)=v$ is added as the unit clause $I_{rcv}$. All these formulas are already in CNF, so they can be added directly with `tell()`.

The definite representation uses `PropDefiniteKB`. A Horn clause has at most one positive literal; a definite Horn clause, as required by this class, has exactly one. It can therefore be written as a conjunction of positive premises implying one positive conclusion. The general at-least-one clause has several positive literals and cannot be inserted directly. I introduce a second family of atoms, $N_{rcv}$, represented by `Not` symbols. Its intended meaning is that value $v$ has been excluded from cell $(r,c)$. Despite its name, `Not2_3_8` is a positive atomic symbol, not the logical expression `~Is2_3_8`; the library does not automatically equate them.

I add the givens as `Is` facts and express exclusions as definite rules. If a cell is $v$, every other value $w$ is excluded from that cell: $I_{rcv}\Rightarrow N_{rcw}$ for $w\ne v$. If another cell $(r',c')$ shares its row, column or box, I add $I_{rcv}\Rightarrow N_{r'c'v}$. To determine a value after eliminating alternatives, I add the last-candidate rule

$$
\left(\bigwedge_{w\in\{1,\ldots,n\}\setminus\{v\}} N_{rcw}\right)
\Rightarrow I_{rcv}.
$$

For a 9-by-9 puzzle, its eight exclusion premises must all be proved before the remaining value can be inferred. For example, the given `Is2_7_8` can establish `Not2_3_8` through the row rule. These rules use the intended exactly-one-value property of Sudoku to make sound deductions from valid givens. They provide an operational elimination encoding, not a logically equivalent replacement for every formula in the general KB: in particular, the two symbol families are independent atoms, and the Horn rules do not explicitly require an unresolved cell to choose some value. Consequently, completeness of Horn inference applies to this rule set, not to all possible Sudoku deductions. The current rules solve all five supplied puzzles, but harder puzzles can require additional rules or search.

On the first puzzle, the general KB contains 11,775 clauses and the definite KB contains 24,087 entries. Some row and box exclusions repeat the same rule. `_prepare_inference` removes these duplicates before inference, leaving 21,171 distinct definite clauses without changing their logical consequences.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Answer:**

I would not use either supplied general-purpose algorithm as the default solver for these 9-by-9 puzzles. Soundness means that an inferred conclusion follows from the KB, and completeness means that an entailed conclusion can be found. Neither property guarantees that the computation will finish within a practical time or memory budget.

The general encoding has $9\times9\times9=729$ propositional symbols. The supplied truth-table procedure recursively assigns True or False to each symbol and checks the KB at complete assignments. Its worst-case search space therefore contains $2^{729}$ assignments, with additional work to evaluate the clauses. Even assignments that do not represent valid Sudoku boards are part of this Boolean search space. The procedure can stop early if it finds a counterexample, but a positive entailment query may require exhausting the search. Its main problem here is the exponential number of assignments to examine.

Resolution instead adds the negated query to the KB and searches for a contradiction. In `logic_.py`, `pl_resolution` constructs a list of all clause pairs on each round, computes their resolvents and keeps newly generated clauses. The first puzzle has 11,775 KB clauses. For the atomic query `Is1_1_1`, adding its negation gives 11,776 initial entries and $\binom{11776}{2}=69,331,200$ initial pairs. Producing that pair list already consumes substantial memory. As new resolvents accumulate, both the clause collection and the number of pairs can grow further; the space of possible clauses is exponential in the number of symbols. Thus resolution suffers from large intermediate clause sets and repeated pair processing, rather than explicitly enumerating truth assignments.

In my recorded experiment, neither method returned an answer before its separate 10-second process limit. The total elapsed times were about 10.12 seconds for resolution and 10.01 seconds for truth-table checking. That limit includes process startup and KB construction, so it is not an exact measurement of inference time alone. A timeout is an inconclusive experiment, not a `False` entailment result. For the supplied puzzles, the restricted Horn representation and chaining algorithms produce the complete boards in practical time. This practical advantage depends on the chosen rules being sufficient for these puzzles.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Answer:**

My backward-chaining function starts with a requested proposition and looks for rules that conclude it. `_prepare_inference` creates an index from each conclusion to its possible premise lists. It also initializes `proven` with the given facts, reuses previous proofs for an unchanged KB, and clears the cached state if the KB's clauses change. Within a rule, every premise must be proved (AND). Across alternative rules for the same conclusion, one successful rule is sufficient (OR). Rules with fewer currently unproved premises are attempted first; this changes search order, not the meaning of a rule.

The following pseudocode follows the implementation. `active` and `attempted` are temporary sets shared by recursive calls within one pass. `proven`, `unprovable` and `proofs` persist while the KB is unchanged.

```text
BACKWARD-ENTAILS(KB, query):
    prepare or reuse the indices and caches for this KB
    if query is in proven: return TRUE
    if query is in unprovable: return FALSE

    repeat:
        previous_size <- size of proven
        active <- empty set
        attempted <- empty set

        if PROVE(query): return TRUE
        if size of proven equals previous_size:
            add attempted to unprovable
            return FALSE
        otherwise repeat with fresh active and attempted sets

PROVE(goal):
    if goal is in proven: return TRUE
    if goal is in unprovable, active, or attempted: return FALSE

    add goal to active
    find every rule whose conclusion is goal
    order those rules by their number of unproved premises
    for each such rule:                         // OR across rules
        recursively PROVE each premise         // AND within this rule
        stop checking this rule if a premise fails
        if all premises succeed:
            remove goal from active
            add goal to proven
            record this rule's premises in proofs[goal]
            return TRUE

    remove goal from active
    add goal to attempted
    return FALSE
```

Cycles are the main correctness challenge. If A depends on B and B depends on A, neither proposition is established just because it appears again in the recursion. The `active` set detects this situation and stops that branch. However, a failed branch is not immediately recorded as permanently unprovable. For example, suppose the rules are $B\Rightarrow A$, $C\Rightarrow A$ and $A\Rightarrow B$, with C a fact. If I tried $B\Rightarrow A$ first while proving A, I would encounter a cycle; the alternative rule from C could still establish A and allow B to be proved later. In this example, my implementation actually tries $C\Rightarrow A$ first because C is already a fact, so it avoids the cyclic branch. The hypothetical order illustrates why permanently caching a failure caused by a cycle could lose a valid proof. My implementation therefore keeps failures temporary within a pass and retries them when the pass has proved new propositions. Only a failed pass with no growth in `proven` makes its failed goals permanently unprovable for that unchanged KB.

Soundness follows because the initial proved propositions are facts, and a new proposition is added only after every premise of an actual rule has been proved. No conclusion is justified by a cycle alone. For termination, the KB has finitely many atoms and rules; this Sudoku vocabulary has at most $2n^3=1,458$ atoms when $n=9$. Within a pass, active goals cannot recur along the same recursion path, and failed goals are not repeatedly expanded. Across passes, `proven` only grows, so only finitely many passes can add new propositions before success or a stable failure. Retrying temporary failures allows grounded proofs through cycles to be found, giving completeness for the encoded finite definite KB. `False` means that the query is not entailed by this KB; it does not by itself prove the query's logical negation.

The saved verification checks all 729 cell/value queries for each of the five supplied puzzles, for a total of 3,645 queries. Each correct value returns `True` and every alternative returns `False`. Both full-grid solvers also match the reference solutions. These are empirical checks on the supplied puzzles; they support the implementation but are separate from the soundness and termination arguments.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Answer:**

I would encode Naked Pairs. Consider two distinct cells p and q in the same row, column or box, and two distinct values a and b. If both cells have been proved unable to take every value outside $\{a,b\}$, then in any valid Sudoku completion their values must occupy a and b in some order. Every other cell in that unit must therefore exclude both values. The reasoning uses established exclusions, rather than treating a value as excluded merely because its `Is` proposition has not been proved.

Write $N_{p,v}$ for the existing `Not` atom for cell p and value v, and let

$$
E = \bigwedge_{v\in\{1,\ldots,n\}\setminus\{a,b\}}
       \left(N_{p,v}\land N_{q,v}\right).
$$

Here E is only an abbreviation for the displayed conjunction, not a new propositional symbol. For each other cell t in the same unit, I would add two separate rules:

$$
E\Rightarrow N_{t,a}
\qquad\text{and}\qquad
E\Rightarrow N_{t,b}.
$$

Each rule has only positive `Not` atoms as premises and exactly one positive `Not` atom as its conclusion. After expanding E, both rules are definite Horn clauses accepted by `PropDefiniteKB`. No disjunctive conclusion, negative premise or test for the absence of a fact is required. For example, if cells (1,1) and (1,2) both exclude 1, 2, 4, 5, 6, 8 and 9, they are restricted to 3 and 7. The fourteen exclusion premises then imply `Not1_5_3` and, through a separate rule, `Not1_5_7`. This conclusion is sound under the Sudoku exactly-one and row-uniqueness constraints, provided the original exclusions are sound.

Adding these rules lets chaining make eliminations before either member of the pair has an individually determined value. Those new exclusions can trigger the existing last-candidate rules. It therefore increases what this particular Horn rule set can deduce, while leaving the chaining procedures unchanged. It does not make the representation a complete solver for every Sudoku or add a general ability to choose between alternative values.

The cost is a larger rule base. A naive 9-by-9 construction considers 27 units, $\binom{9}{2}$ cell pairs, $\binom{9}{2}$ value pairs and fourteen possible single-atom conclusions per pattern. That gives $27\times36\times36\times14=489,888$ additional grounded rules before duplicate removal, each with fourteen premises. Building, indexing and searching these rules can substantially increase memory and runtime. Generating only applicable patterns could reduce that cost, but would require additional implementation work. This answer describes a possible extension; the current `build_definite_kb` implements basic elimination and last-candidate rules, not Naked Pairs.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Answer:**

Backward chaining is attractive when the KB is large but only a small part of it is relevant to the requested conclusion. For example, a user may ask about one particular cell whose proof depends on a few givens and elimination rules, while many other rules concern unrelated cells or units. With a conclusion index, backward chaining can follow those few dependencies instead of processing all consequences of the available facts. Repeated queries can also reuse established proofs. In my implementation, however, the first call builds indices for the whole KB and later calls check the clause snapshot. Those preparation costs are real: the strongest benefit occurs when they are reused or are small compared with the inference work saved. Forward chaining can also stop when it reaches its query, so its extra work depends on the order in which facts are processed.

Backward chaining may offer little advantage when the goal depends on most of the KB, when many alternative proofs fail, or when cyclic dependencies require repeated passes. Querying every cell and value can also make the relevant portion approach the whole rule base. In that situation, forward chaining can be preferable if it computes the Horn consequences once and then answers many queries from the resulting closure. Backward caching reduces repeated work, but does not eliminate rule lookup, recursion, alternative-rule exploration and cycle-management costs. Sudoku dependencies can spread through shared rows, columns and boxes, so a single cell query is not guaranteed to remain local.

In my recorded single-puzzle run, `solve_full_grid_fc` took 5.9964 seconds and `solve_full_grid_bc` took 0.2081 seconds on the same 30-given puzzle, with both returning the correct 81-cell solution. These timings include KB construction. The roughly 28.8-fold difference describes these implementations on this run; it is not a general claim that backward chaining is always faster.

The implementation details explain much of that gap. My forward solver uses the supplied `pl_fc_entails` and an indexed `clauses_with_premise` lookup, skips givens and adds each proved cell value as a fact. Nevertheless, each candidate query invokes `pl_fc_entails` again, rebuilding its premise counts, agenda and inferred flags. The backward solver keeps the same KB and reuses proved and unprovable goals across candidate queries. Thus this experiment compares repeated forward queries with a cached backward procedure, rather than a single forward-closure pass with an uncached backward search. The five-puzzle verification confirms correct results for both methods, but a broader benchmark and matched reuse strategies would be needed to compare the algorithms more generally.

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`
